# Run City Scan Tasks Locally

Run tasks 6, 7, 14, and 16 locally and save outputs directly to `mnt/` city folders.

## 1. Select City

In [1]:
import os
import sys
import yaml
import geopandas as gpd

# List available cities in mnt/
cities = [d for d in os.listdir('mnt') if os.path.isdir(f'mnt/{d}') and d.startswith('20')]
print("Available cities:")
for i, city in enumerate(cities, 1):
    print(f"{i}. {city}")


Available cities:
1. 2025-10-senegal-tambacounda
2. 2025-10-senegal-diourbel
3. 2025-02-tunisia-tunis
4. 2025-10-indonesia-sofifi
5. 2025-10-senegal-dakar
6. 2025-10-senegal-matam


In [2]:
# Select city
city_dir = "mnt/2025-10-senegal-matam"  # Change this to your city

print(f"\nSelected: {city_dir}")


Selected: mnt/2025-10-senegal-matam


## 2. Load City Configuration

In [3]:
# Load city_inputs.yml from city folder
with open(f'{city_dir}/01-user-input/city_inputs.yml', 'r') as f:
    city_inputs = yaml.safe_load(f)

# Load menu.yml
with open(f'{city_dir}/01-user-input/menu.yml', 'r') as f:
    menu = yaml.safe_load(f)

city_name = city_inputs['city_name']
city_name_l = city_name.replace(' ', '_').replace("'", "").lower()
country_iso3 = "SEN"  # Change for different countries

print(f"City: {city_name}")
print(f"Country ISO3: {country_iso3}")

City: Matam
Country ISO3: SEN


## 3. Load AOI

In [4]:
# Load AOI from city's 01-user-input folder
aoi_shp_name = city_inputs['AOI_shp_name']
aoi_path = f"{city_dir}/01-user-input/AOI/{aoi_shp_name}.shp"
aoi_file = gpd.read_file(aoi_path).to_crs(epsg=4326)

print(f"✓ AOI loaded: {len(aoi_file)} features")

✓ AOI loaded: 1 features


## 4. Setup Output Directories

In [5]:
# Output directories
output_spatial = f"{city_dir}/02-process-output/spatial"
output_tabular = f"{city_dir}/02-process-output/tabular"

os.makedirs(output_spatial, exist_ok=True)
os.makedirs(output_tabular, exist_ok=True)

print(f"Output spatial: {output_spatial}")
print(f"Output tabular: {output_tabular}")

Output spatial: mnt/2025-10-senegal-matam/02-process-output/spatial
Output tabular: mnt/2025-10-senegal-matam/02-process-output/tabular


---
# Tasks

## Task 6: Demographics (WorldPop age/sex)

In [6]:
if menu.get('demographics', False):
    sys.path.insert(0, 'backend-local')
    import demographics_local
    
    demographics_local.run_demographics(city_name_l, country_iso3, aoi_file, output_tabular)
else:
    print("Demographics is disabled in menu.yml")

Running demographics for matam...
Fetching WorldPop age structures for SEN...
Found 36 age/sex raster files
  sen_m_25_2020_constrained.tif already exists, skipping
  sen_f_15_2020_constrained.tif already exists, skipping
  sen_f_5_2020_constrained.tif already exists, skipping
  sen_f_30_2020_constrained.tif already exists, skipping
  sen_m_30_2020_constrained.tif already exists, skipping
  sen_f_10_2020_constrained.tif already exists, skipping
  sen_m_40_2020_constrained.tif already exists, skipping
  sen_m_35_2020_constrained.tif already exists, skipping
  sen_f_40_2020_constrained.tif already exists, skipping
  sen_f_45_2020_constrained.tif already exists, skipping
  sen_f_0_2020_constrained.tif already exists, skipping
  sen_f_70_2020_constrained.tif already exists, skipping
  sen_m_1_2020_constrained.tif already exists, skipping
  sen_m_70_2020_constrained.tif already exists, skipping
  sen_f_50_2020_constrained.tif already exists, skipping
  sen_f_60_2020_constrained.tif already 

## Task 7: Population (WorldPop)

In [7]:
if menu.get('population', False):
    sys.path.insert(0, 'backend-local')
    import population_local
    
    population_local.run_population(city_name_l, country_iso3, aoi_file, output_spatial)
else:
    print("Population is disabled in menu.yml")

Running WorldPop population for matam...
Fetching WorldPop data for SEN...
Found 1 raster files
Mosaicking rasters...
Clipping to AOI...
✓ Population raster saved to: mnt/2025-10-senegal-matam/02-process-output/spatial/matam_population.tif


## Task 14: Relative Wealth Index (RWI)

**Data Source:** Meta/Facebook Relative Wealth Index (automatically downloaded from GCS)

In [8]:
if menu.get('rwi', False):
    sys.path.insert(0, 'backend-local')
    import rwi_local
    
    rwi_local.run_rwi(city_name_l, country_iso3, aoi_file, output_spatial)
else:
    print("RWI is disabled in menu.yml")

Running RWI for matam...
Using cached RWI data from data/rwi/SEN_relative_wealth_index.csv
Loading RWI data...
Converting 14392 quadkeys to polygons...
Clipping to AOI...
✓ RWI data saved to: mnt/2025-10-senegal-matam/02-process-output/spatial/matam_rwi.gpkg
  10 tiles within AOI
  RWI range: -0.595 to 0.333


---
## Task 16: GEE Outputs

**Important:** GEE exports to Google Drive. You need to:
1. Run the cells below to start export tasks
2. Check https://code.earthengine.google.com/tasks for progress
3. Download files from Google Drive folder `city-scan-outputs`
4. Manually move them to your `mnt/{city}/02-process-output/spatial/` folder

### Initialize Earth Engine

In [9]:
import ee

ee.Authenticate()
# Initialize Earth Engine
ee.Initialize(project = "acr-dev-450519")

### Run GEE Tasks

In [10]:
sys.path.insert(0, 'backend-local')
import gee_local

first_year = city_inputs.get('first_year', 2015)
last_year = city_inputs.get('last_year', 2024)

print("Starting GEE export tasks...\n")

# Forest
if menu.get('forest', False):
    gee_local.gee_forest(city_name_l, aoi_file)

# Green (NDVI)
if menu.get('green', False):
    gee_local.gee_ndvi(city_name_l, aoi_file, first_year, last_year)

# Landcover
if menu.get('landcover', False):
    gee_local.gee_landcover(city_name_l, aoi_file)

# LST Summer
if menu.get('lst_summer', False):
    gee_local.gee_lst_summer(city_name_l, aoi_file, first_year, last_year)

# LST Winter
if menu.get('lst_winter', False):
    gee_local.gee_lst_winter(city_name_l, aoi_file, first_year, last_year)

# NDMI
if menu.get('ndmi', False):
    gee_local.gee_ndmi(city_name_l, aoi_file, first_year, last_year)

# Nightlight
if menu.get('nightlight', False):
    gee_local.gee_nightlight(city_name_l, aoi_file)

print("\n✓ All export tasks started!")
print("\nNext steps:")
print("1. Go to https://code.earthengine.google.com/tasks")
print("2. Wait for tasks to complete (may take 10-30 minutes)")
print("3. Download files from Google Drive folder: city-scan-outputs")
print(f"4. Move files to: {output_spatial}/")

Starting GEE export tasks...

Running gee_forest - exporting to Google Drive...


/Users/vivaldirinaldi/miniforge3/envs/cityscan/lib/python3.10/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2023_v1_11! You are using a deprecated asset.
To ensure continued functionality, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11

  warnings.warn(warning, category=DeprecationWarning)


✓ Tasks started: matam_forest_cover23, matam_deforestation
  Files will be saved to Google Drive: city-scan-outputs/matam/
  Check https://code.earthengine.google.com/tasks for progress
Running gee_ndvi - exporting to Google Drive...
✓ Task started: matam_ndvi_season
  Files will be saved to Google Drive: city-scan-outputs/matam/
  Check https://code.earthengine.google.com/tasks for progress
Running gee_landcover - exporting to Google Drive...
✓ Task started: matam_lc
  Files will be saved to Google Drive: city-scan-outputs/matam/
  Check https://code.earthengine.google.com/tasks for progress
Running gee_lst_summer - exporting to Google Drive...
✓ Task started: matam_summer
  Files will be saved to Google Drive: city-scan-outputs/matam/
  Check https://code.earthengine.google.com/tasks for progress
Running gee_lst_winter - exporting to Google Drive...
✓ Task started: matam_winter
  Files will be saved to Google Drive: city-scan-outputs/matam/
  Check https://code.earthengine.google.com